In [7]:
import numpy as np
import pandas as pd

# Modelos
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor, StackingRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import xgboost as xgb
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Ruta de archivo de datos
ruta = 'data/datos_viejos.csv'
df = pd.read_csv(ruta)

# Definir la variable objetivo y las características
target = 'C4'
features = [
    'Especie', 'Ocupa3', 'Estado3', 'FPMasa', 'FccTot', 'FccArb', 'DisEsp', 'ComEsp',
    'Rocosid', 'Textura', 'MatOrg', 'TipSuelo1', 'TipSuelo2', 'TipSuelo3', 'MErosiva', 
    'ModComb', 'EspCMue', 'Orienta1', 'Superficie',  'Periodo',
    'NPies153', 'NPies203', 'NPies253', 'NPies303', 'NPies353', 'NPies403', 'NPies453', 
    'NPies503', 'NPies553', 'NPies603', 'NPies653', 'NPies703', 'NPies13', 'NPies23', 
    'NPies53', 'ASP_mean', 'ELE_mean', 'SLO_mean', 'EVI_mean_layer', 'EVI_std_dev_layer', 
    'GNDVI_mean_layer', 'GNDVI_std_dev_layer', 'NDII_mean_layer', 'NDII_std_dev_layer', 
    'NDVI_mean_layer', 'NDVI_std_dev_layer', 'temp_summer_min', 'temp_summer_max', 
    'temp_summer_mean', 'temp_summer_std', 'temp_spring_min', 'temp_spring_max',
    'temp_spring_mean', 'temp_spring_std', 'pr_total_summer', 'pr_total_spring', 
    'pr_total_marzo', 'pr_total_abril', 'pr_total_mayo', 'pr_total_junio', 'pr_total_julio', 
    'pr_total_agosto'
]

# Preprocesamiento de datos
cat_features = ['Especie', 'Estado3','FPMasa', 'DisEsp', 'ComEsp', 'Rocosid',
                'Textura', 'MatOrg','TipSuelo1', 'TipSuelo2', 'TipSuelo3', 'MErosiva', 'ModComb']
num_features = [col for col in features if col not in cat_features]

# Preprocesadores de características numéricas y categóricas
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer para manejar diferentes tipos de características
preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# Modelos base
modelos_base = [
    ('LightGBM', LGBMRegressor(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42)),
    ('XGBoost', xgb.XGBRegressor(random_state=42, verbosity=0)),
    ('CatBoost', CatBoostRegressor(random_state=42, verbose=0)),
    ('GBDT', GradientBoostingRegressor(random_state=42)),
    ('BaggedDT', BaggingRegressor(random_state=42))
]

# Crear el StackingRegressor
stacking_model = StackingRegressor(estimators=modelos_base, final_estimator=SVR(kernel='linear'))

# Dividir los datos en entrenamiento y prueba
df = df.dropna()
df = df[df['C3'] < df['C4']]
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear y entrenar el pipeline con Stacking
stacking_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('stacking', stacking_model)
])

# Entrenar el modelo
stacking_pipeline.fit(X_train, y_train)

# Evaluación del modelo
y_pred = stacking_pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Stacking Model - R2: {r2:.4f}, RMSE: {rmse:.4f}")

# Realizar predicciones en nuevas observaciones
# Nuevas observaciones
nuevas_observaciones = X_test  # sustituir por las nuevas observaciones a predecir

# Predecir utilizando el modelo
predicciones = stacking_pipeline.predict(nuevas_observaciones)
print(predicciones)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.579701 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7724
[LightGBM] [Info] Number of data points in the train set: 3007, number of used features: 94
[LightGBM] [Info] Start training from score 46.186232
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.577854 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7688
[LightGBM] [Info] Number of data points in the train set: 2405, number of used features: 92
[LightGBM] [Info] Start training from score 46.015756


<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001865 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7692
[LightGBM] [Info] Number of data points in the train set: 2405, number of used features: 92
[LightGBM] [Info] Start training from score 46.712648


<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007565 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7683
[LightGBM] [Info] Number of data points in the train set: 2406, number of used features: 92
[LightGBM] [Info] Start training from score 46.037086


<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


... (2 outputs left out)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000895 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7692
[LightGBM] [Info] Number of data points in the train set: 2406, number of used features: 92
[LightGBM] [Info] Start training from score 45.830128


<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Stacking Model - R2: 0.8075, RMSE: 18.0286
[ 6.92810473e+00  2.93707169e+01  5.44140348e+01  3.68927306e+01
  2.65451754e+01  1.24060517e+02  1.94991128e+01  5.17826973e+01
  2.15986621e+01  8.27353410e+01  2.06928066e+01  2.54642120e+01
  6.66599029e+00  4.71686500e+01  4.62210675e+01  3.67007192e+00
  2.05491461e+01  1.17664394e+01  1.15855718e+01  4.62349526e+01
  4.12516481e+01  4.43499111e+01  1.12063981e+02  2.81783201e+01
  5.70990264e+01  2.25929181e+01  2.02116887e+01  4.12270497e+01
  2.79870029e+01  5.90128547e+01  2.40616059e+01  1.65227449e+01
  8.28249903e+01  1.97009988e+01  1.93452522e+01  6.24303422e+00
  4.49239808e+01  1.54204973e+01  1.50521668e+01  9.77583517e+00
  1.21902795e+02  4.16114411e+01  3.93311866e+01  3.45600299e+01
  2.43334476e+01  3.09164601e+01  1.19077215e+02  9.25574125e+00
  1.34804884e+02  1.83361950e+01  3.72620791e+01  3.38534005e+01
  1.93448220e+01  1.02943853e+01  1.91344549e+01  3.78422370e+01
  2.96674061e+01  2.45167749e+01  3.51321386e+0

<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [6]:
import pandas as pd

# Cargar el archivo
ruta = 'data/datos_viejos.csv'  # Cambia la ruta según corresponda
df = pd.read_csv(ruta)

# Filtrar y contar las entradas que cumplen con la condición
num_entries = len(df[df['C3'] < df['C4']])
print(f"Cantidad de entradas donde C3 < C4: {num_entries}")


Cantidad de entradas donde C3 < C4: 3788


In [8]:
import joblib

# Guardar el modelo entrenado
joblib.dump(stacking_pipeline, 'stacking_model.pkl')

# Cargar el modelo cuando lo necesites
loaded_model = joblib.load('stacking_model.pkl')

# Predicciones con el modelo cargado
predicciones = loaded_model.predict(nuevas_observaciones)
print(predicciones)


<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[ 6.92810473e+00  2.93707169e+01  5.44140348e+01  3.68927306e+01
  2.65451754e+01  1.24060517e+02  1.94991128e+01  5.17826973e+01
  2.15986621e+01  8.27353410e+01  2.06928066e+01  2.54642120e+01
  6.66599029e+00  4.71686500e+01  4.62210675e+01  3.67007192e+00
  2.05491461e+01  1.17664394e+01  1.15855718e+01  4.62349526e+01
  4.12516481e+01  4.43499111e+01  1.12063981e+02  2.81783201e+01
  5.70990264e+01  2.25929181e+01  2.02116887e+01  4.12270497e+01
  2.79870029e+01  5.90128547e+01  2.40616059e+01  1.65227449e+01
  8.28249903e+01  1.97009988e+01  1.93452522e+01  6.24303422e+00
  4.49239808e+01  1.54204973e+01  1.50521668e+01  9.77583517e+00
  1.21902795e+02  4.16114411e+01  3.93311866e+01  3.45600299e+01
  2.43334476e+01  3.09164601e+01  1.19077215e+02  9.25574125e+00
  1.34804884e+02  1.83361950e+01  3.72620791e+01  3.38534005e+01
  1.93448220e+01  1.02943853e+01  1.91344549e+01  3.78422370e+01
  2.96674061e+01  2.45167749e+01  3.51321386e+01  3.09058849e+01
  2.07878796e+01  7.26445

In [16]:
import pandas as pd
import numpy as np

# Variables de interés
variables = [
    'Especie', 'Ocupa3', 'Estado3', 'FPMasa', 'FccTot', 'FccArb', 'DisEsp', 'ComEsp',
    'Rocosid', 'Textura', 'MatOrg', 'TipSuelo1', 'TipSuelo2', 'TipSuelo3', 'MErosiva', 
    'ModComb', 'EspCMue', 'Orienta1', 'Superficie', 'Periodo',
    'NPies153', 'NPies203', 'NPies253', 'NPies303', 'NPies353', 'NPies403', 'NPies453', 
    'NPies503', 'NPies553', 'NPies603', 'NPies653', 'NPies703', 'NPies13', 'NPies23', 
    'NPies53', 'ASP_mean', 'ELE_mean', 'SLO_mean', 'EVI_mean_layer', 'EVI_std_dev_layer', 
    'GNDVI_mean_layer', 'GNDVI_std_dev_layer', 'NDII_mean_layer', 'NDII_std_dev_layer', 
    'NDVI_mean_layer', 'NDVI_std_dev_layer', 'temp_summer_min', 'temp_summer_max', 
    'temp_summer_mean', 'temp_summer_std', 'temp_spring_min', 'temp_spring_max',
    'temp_spring_mean', 'temp_spring_std', 'pr_total_summer', 'pr_total_spring', 
    'pr_total_marzo', 'pr_total_abril', 'pr_total_mayo', 'pr_total_junio', 'pr_total_julio', 
    'pr_total_agosto'
]

# Opciones explícitas
options = pd.DataFrame({ 
    'Especie': [23, 26, 45, 71],
    'Ocupa3': [10,10,10,10],
    'Estado3': [1, 1, 1, 1],
    'FPMasa': [1, 1, 1, 1],
    'FccTot': [90, 90, 90, 90],
    'FccArb': [100, 100, 100, 100],
    'DisEsp': [1, 1, 1, 1],
    'ComEsp': [1, 1, 1, 1],
    'Superficie': [1, 1, 1, 1],
    'Periodo': [17, 17, 17, 17],
    'NPies53': [500, 500, 500, 500],
})


# Suponemos que 'df' ya está cargado y contiene todas las variables necesarias

# Variables que empiezan por NPies (completaremos con 0 si no están en options)
npies_vars = [col for col in variables if col.startswith('NPies')]
# Ya NPies53 está en options, el resto poner a 0

# Extraemos observaciones con max y min C4
df_max = df.loc[df['C4'].idxmax()]
df_min = df.loc[df['C4'].idxmin()]

# Variables que faltan en options y no NPies
faltan = [v for v in variables if v not in options.columns and v not in npies_vars]

# Creamos DataFrames para valores max y min para las variables faltantes
max_vals = pd.DataFrame({v: [df_max[v]]*len(options) for v in faltan})
min_vals = pd.DataFrame({v: [df_min[v]]*len(options) for v in faltan})

# Creamos DataFrames para NPies no definidas en options, con valor 0
npies_0 = pd.DataFrame({v: [0]*len(options) for v in npies_vars if v not in options.columns})

# Combinamos todo para max y min
df_max_comb = pd.concat([options.reset_index(drop=True), max_vals.reset_index(drop=True), npies_0.reset_index(drop=True)], axis=1)
df_min_comb = pd.concat([options.reset_index(drop=True), min_vals.reset_index(drop=True), npies_0.reset_index(drop=True)], axis=1)

# Número de combinaciones = len(options) * 2 (max y min)
# Finalmente, combinamos filas max y min para obtener todas las combinaciones
result = pd.concat([df_max_comb, df_min_comb], ignore_index=True)

# Para obtener todas combinaciones cruzadas (8 filas: 4 opciones * 2 valores max/min)
# Aquí se repite options dos veces, alternando max/min para las variables faltantes

print(result)


   Especie  Ocupa3  Estado3  FPMasa  FccTot  FccArb  DisEsp  ComEsp  \
0       23      10        1       1      90     100       1       1   
1       26      10        1       1      90     100       1       1   
2       45      10        1       1      90     100       1       1   
3       71      10        1       1      90     100       1       1   
4       23      10        1       1      90     100       1       1   
5       26      10        1       1      90     100       1       1   
6       45      10        1       1      90     100       1       1   
7       71      10        1       1      90     100       1       1   

   Superficie  Periodo  ...  NPies353  NPies403  NPies453  NPies503  NPies553  \
0           1       17  ...         0         0         0         0         0   
1           1       17  ...         0         0         0         0         0   
2           1       17  ...         0         0         0         0         0   
3           1       17  ...         

In [18]:
import pandas as pd
import joblib  # para cargar el modelo
import numpy as np

# Cargar modelo guardado
modelo_path = 'models/stacking_model.pkl'
model = joblib.load(modelo_path)

# Suponemos que 'result' ya está creado con las combinaciones de variables para predecir

# Predecir con el modelo
# Si el modelo es un Pipeline con preprocesamiento incluido, se puede predecir directo
preds = model.predict(result)

# Añadir columna de predicción al DataFrame
result['Prediccion'] = preds

# Ordenar de mayor a menor según la predicción
result_sorted = result.sort_values(by='Prediccion', ascending=False).reset_index(drop=True)

# Añadir columna ranking
result_sorted['Ranking'] = np.arange(1, len(result_sorted) + 1)

vars_to_print = [
    'Especie', 'Rocosid', 'Textura', 'MatOrg', 'TipSuelo1', 'TipSuelo2', 'TipSuelo3', 
    'MErosiva', 'ModComb', 'EspCMue', 'Orienta1', 'Superficie', 'Periodo'
]

print(result_sorted[['Ranking', 'Prediccion'] + vars_to_print])



   Ranking  Prediccion  Especie  Rocosid  Textura  MatOrg  TipSuelo1  \
0        1   67.702453       45      3.0      2.0     1.0        1.0   
1        2   67.590030       71      3.0      2.0     1.0        1.0   
2        3   67.579894       23      3.0      2.0     1.0        1.0   
3        4   65.126965       26      3.0      2.0     1.0        1.0   
4        5   43.396776       45      2.0      1.0     1.0        1.0   
5        6   43.248954       71      2.0      1.0     1.0        1.0   
6        7   43.123914       23      2.0      1.0     1.0        1.0   
7        8   40.374770       26      2.0      1.0     1.0        1.0   

   TipSuelo2  TipSuelo3  MErosiva  ModComb  EspCMue  Orienta1  Superficie  \
0        2.0        1.0       1.0      9.0      3.0     250.0           1   
1        2.0        1.0       1.0      9.0      3.0     250.0           1   
2        2.0        1.0       1.0      9.0      3.0     250.0           1   
3        2.0        1.0       1.0      9.0 

<venv>\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
